In [63]:
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

In [64]:
df=yf.download("IGE",start="2001-01-01",end="2007-01-01")
df.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,IGE,IGE,IGE,IGE,IGE
Date,,,,,
2001-11-26,9.153774,9.153774,9.153774,9.153774,0
2001-11-27,9.153774,9.153774,9.153774,9.153774,0
2001-11-28,9.153774,9.153774,9.153774,9.153774,0
2001-11-29,9.153774,9.153774,9.153774,9.153774,0
2001-11-30,9.184953,9.184953,9.184953,9.184953,600


### Daily Returns

In [65]:
dailyret = df["Close"]["IGE"].pct_change().dropna()

### Excess returns

In [66]:
excessRet = dailyret - 0.04/245 # excess dailt returns=  strategy returns - financing cost, assuming risk-free rate

In [67]:
sharpeRatio = np.sqrt(252) * np.mean(excessRet) / np.std(excessRet)

print(f"Sharpe Ratio (Long-Only): {sharpeRatio:.4f}")

Sharpe Ratio (Long-Only): 0.7189


## **Market Neutral**

In [68]:
spy = yf.download("SPY",start="2001-01-01",end="2007-01-01")
spy.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,SPY,SPY,SPY,SPY,SPY
Date,,,,,
2001-01-02,81.774338,83.897058,80.980797,83.797866,8737500
2001-01-03,85.702370,86.337202,81.040319,81.456928,19431600
2001-01-04,84.779854,85.999922,84.432680,85.662667,9219000
2001-01-05,82.012367,84.829435,82.012367,84.730242,12911400
2001-01-08,82.647263,82.647263,81.060181,82.448877,6625300


In [69]:
marketRet = spy["Close"]["SPY"].pct_change().dropna()

In [70]:
dailyret,marketRet = dailyret.align(marketRet,join="inner")

In [71]:
netRet = (dailyret - marketRet) / 2

In [72]:
sharpeRatio_neutral = np.sqrt(252) * np.mean(netRet) / np.std(netRet)

print(f"Sharpe Ratio (Market Neutral): {sharpeRatio_neutral:.4f}")

Sharpe Ratio (Market Neutral): 0.6745


# **Drawdown**

In [73]:
dailyret2 = df["Close"]["IGE"].pct_change().dropna()

## compound cummulative returns

In [74]:
cumret = (1+dailyret).cumprod() - 1

## Max Drawdown

In [75]:
def calculateMaxDD(current):
    cumret = current.values
    highwatermark = np.zeros(cumret.shape)
    drawdown = np.zeros(cumret.shape)
    drawdownduration = np.zeros(cumret.shape)

    for t in np.arange(1,cumret.shape[0]):
        highwatermark[t] = np.maximum(highwatermark[t-1],cumret[t])
        drawdown[t] = (1+current[t])/(1+highwatermark[t])-1
        if drawdown[t] == 0:
            drawdownduration[t] = 0
        else:
            drawdownduration[t] = drawdownduration[t-1]+1

    maxDD, i = np.min(drawdown), np.argmin(drawdown)
    maxDDD = np.max(drawdownduration)
    return maxDD, maxDDD, i

In [76]:
maxDD, maxDDD, i = calculateMaxDD(cumret)

print(f"Max Drawdown: {maxDD:.4f} ({maxDD*100:.2f}%)")
print(f"Max Drawdown Duration: {maxDDD} days")

Max Drawdown: -0.3146 (-31.46%)
Max Drawdown Duration: 432.0 days


/tmp/ipykernel_4906/731434839.py:9: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  drawdown[t] = (1+current[t])/(1+highwatermark[t])-1
